# Final Integrated Challenge


## Scenario

You are building a model that predicts whether a person's annual income exceeds **$50K**, using the **Adult Census Income** dataset from OpenML:

```python
from sklearn.datasets import fetch_openml

adult = fetch_openml(name="adult", version=2, as_frame=True)

Your task is to build a fully reproducible Scikit-learn training pipeline, track every experiment with
MLflow, and manage the project history with Git — bringing together everything from the last three
sessions into one complete workflow

# Libraries

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

# Data Loading

In [10]:
adult = fetch_openml(name='adult', version=2, as_frame=True)

X = pd.DataFrame(data=adult.data)
y = pd.Series(adult.target, name=adult.target_names[0])

In [11]:
X.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


In [12]:
def data_summary(df: pd.DataFrame, target_col: pd.Series) -> pd.DataFrame:
    """Prints basic data diagnostics and returns missing value summary.

    Args:
        df: Input DataFrame to analyze.
        target_col: Name of the target column to show class distribution for.

    Returns:
        DataFrame of columns with missing values, sorted by missing percentage.
    """
    df = df.copy()
    print('Data shape:')
    print(f'Number of rows: {df.shape[0]}, Number of columns: {df.shape[1]}')
    print("="* 100 )
    print("Target value distribution:")
    print(target_col.value_counts(normalize=True))
    print("="* 100 )
    missing_values = (df.isna().mean() * 100).round(2)
    col_types = df.dtypes
    unique_values = df.nunique()
    summary_df = pd.DataFrame({
        'perc_missing_values' : missing_values,
        'col_type' : col_types,
        'number_unique_values' : unique_values
    })
    return summary_df.sort_values(by='perc_missing_values', ascending=False)



data_summary(df=X, target_col=y)

Data shape:
Number of rows: 48842, Number of columns: 14
Target value distribution:
class
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64


,perc_missing_values,col_type,number_unique_values
occupation,5.75,category,14
workclass,5.73,category,8
native-country,1.75,category,41
education,0.00,category,16
fnlwgt,0.00,int64,28523
age,0.00,int64,74
marital-status,0.00,category,7
education-num,0.00,int64,16
race,0.00,category,5
relationship,0.00,category,6


In [13]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
dtypes: category(8), int64(6)
memory usage: 2.6 MB


In [14]:
X.describe().T

,count,mean,std,min,25%,50%,75%,max
age,48842.0,38.643585,13.710510,17.0,28.0,37.0,48.0,90.0
fnlwgt,48842.0,189664.134597,105604.025423,12285.0,117550.5,178144.5,237642.0,1490400.0
education-num,48842.0,10.078089,2.570973,1.0,9.0,10.0,12.0,16.0
capital-gain,48842.0,1079.067626,7452.019058,0.0,0.0,0.0,0.0,99999.0
capital-loss,48842.0,87.502314,403.004552,0.0,0.0,0.0,0.0,4356.0
hours-per-week,48842.0,40.422382,12.391444,1.0,40.0,40.0,45.0,99.0


In [15]:
numerical_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.select_dtypes(exclude=np.number).columns.tolist()

# Data Splitting

In [16]:
target_class = {'<=50K' : 0, '>50K' : 1}
y = y.map(target_class)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

len(X_train), len(y_train), len(X_test), len(y_test)

(39073, 39073, 9769, 9769)

In [20]:
numerical_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="median")),
    ('scale', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='constant', fill_value="missing")),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocess = ColumnTransformer(transformers=[
    ('num_preprocess', numerical_pipeline, numerical_columns),
    ('cat_preprocess', categorical_pipeline, categorical_columns)
])
